# 02 · LoRA fine-tune `nvidia/Nemotron-Mini-4B-Instruct`

**Switch this runtime to a T4 GPU** (Runtime → Change runtime type → T4 GPU) before running.

Loads the base model in 4-bit, attaches a LoRA adapter via `peft`, trains with `trl.SFTTrainer` on the split saved by `01_prepare_dataset.ipynb`, then pushes just the adapter to your own Hugging Face namespace so it can be loaded on top of the base model at serve time by vLLM or SGLang.

In [ ]:
!pip install -q "transformers>=4.44" "peft>=0.12" "trl>=0.9" "accelerate>=0.33" "bitsandbytes>=0.43" datasets huggingface_hub

In [ ]:
from huggingface_hub import login

# Needs a WRITE-scoped token this time, since this notebook pushes the adapter
# to your own namespace: https://huggingface.co/settings/tokens
login()

In [ ]:
import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "nvidia/Nemotron-Mini-4B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
train_ds = load_from_disk("/content/daring_anteater_train")
eval_ds = load_from_disk("/content/daring_anteater_eval")
print(f"train: {len(train_ds)}  eval: {len(eval_ds)}")

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="/content/nemotron-mini-4b-daring-anteater-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    bf16=True,
    max_seq_length=1024,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
metrics = trainer.evaluate()
print(metrics)

In [ ]:
# Push ONLY the adapter (a few hundred MB), not a merged copy of the base model.
# Replace <your-hf-username> below.
HUB_REPO = "<your-hf-username>/nemotron-mini-4b-daring-anteater-lora"

trainer.model.push_to_hub(HUB_REPO)
tokenizer.push_to_hub(HUB_REPO)
print(f"Adapter pushed to https://huggingface.co/{HUB_REPO}")
print("Use this repo id as --lora-modules / --lora-path in the vLLM and SGLang serve scripts.")